Installing Libraries:

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn

Libraries

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

Train

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

# === CONFIGURATION ===
BASE_PATH = "/content/drive/MyDrive/Datasets/Final UTFVP_Original with Augmented Images_4"
IMAGE_SIZE = (128, 60)
FINGER_NUMS = [1, 2, 3, 4, 5, 6]

# Protocol 3 - use: original + 3 augmented from img1,2; 3 augmented only from img3,4
PROTOCOL3_TRAIN_INDICES = {
    1: ['original', 1, 2, 3],
    2: ['original', 1, 2, 3],
    3: [1, 2, 3],
    4: [1, 2, 3]
}

# === FUNCTION: Denoising ===
def apply_denoising(image, h=10):
    print("Denoising image...")
    return cv2.fastNlMeansDenoising(image, h=h)

# === FUNCTION TO FUSE FINGERS FOR TRAINING ===
def fuse_fingers_p3(subject_path, subject_id):
    fused_samples = []
    labels = []

    sample_counter = 0
    for img_num, versions in PROTOCOL3_TRAIN_INDICES.items():
        for ver in versions:
            finger_images = []
            complete = True
            sample_counter += 1
            ver_str = f"{img_num}" if ver == 'original' else f"{img_num}_{ver}_Augmented"
            print(f"\nSubject {subject_id} — Image {ver_str} (Sample #{sample_counter:02d})")

            for finger in FINGER_NUMS:
                if ver == 'original':
                    fname = f"{subject_id}_{finger}_{img_num}.png"
                else:
                    fname = f"{subject_id}_{finger}_{img_num}_{ver}_Augmented.png"

                img_path = os.path.join(subject_path, fname)
                print(f" Loading: {img_path}")

                if not os.path.exists(img_path):
                    print(f" File not found: {img_path}")
                    complete = False
                    break

                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                img = cv2.resize(img, IMAGE_SIZE)
                img = img.astype(np.float32) / 255.0  # Normalize to [0, 1]
                img_denoised = apply_denoising((img * 255).astype(np.uint8), h=10)
                img_denoised = img_denoised.astype(np.float32) / 255.0
                img_eq = exposure.equalize_hist(img_denoised)
                img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)
                finger_images.append(img_norm)

            if complete and len(finger_images) == 6:
                fused = np.vstack(finger_images)
                label = f"{subject_id}_fused_p3_{sample_counter:02d}"
                fused_samples.append(fused)
                labels.append(label)
                print(f" ✅ Created fused sample: {label}")
            else:
                print(f" ❌ Incomplete sample skipped for Subject {subject_id}, Sample #{sample_counter:02d}")

    return fused_samples, labels

# === LOAD TRAINING DATA ===
train_data = []
train_labels = []

print("Scanning dataset folders...")
subject_dirs = sorted(os.listdir(BASE_PATH))
for subj in tqdm(subject_dirs, desc="Loading Subjects"):  # ✅ Fixed emoji issue
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue
    print(f"Processing subject: {subj}")
    samples, labels = fuse_fingers_p3(subject_path, subj)
    train_data.extend(samples)
    train_labels.extend(labels)

train_data = np.array(train_data)
train_labels = np.array(train_labels)

print("\n✅ Final Train Data Loaded")
print(f"   Total Samples: {train_data.shape[0]}")
print(f"   Image Shape: {train_data[0].shape}")
print(f"   Labels Example: {train_labels[:5]}")

# === 2DPCA FUNCTIONS ===
def compute_2dpca(images_2d, num_components):
    print("\n📐 Computing 2DPCA projection matrix...")
    n = len(images_2d)
    h, w = images_2d[0].shape
    mean_img = sum(images_2d) / n
    G_t = np.zeros((w, w))

    for i, img in enumerate(images_2d):
        A = img - mean_img
        G_t += A.T @ A
        if i < 3:
            print(f"   ➕ Added contribution from sample {i + 1}")

    G_t /= n
    eig_vals, eig_vecs = np.linalg.eigh(G_t)
    idx = np.argsort(-eig_vals)
    eig_vecs = eig_vecs[:, idx[:num_components]]
    print(f"✅ 2DPCA matrix W shape: {eig_vecs.shape}")
    return eig_vecs

def project_2dpca(images_2d, W):
    print("\n🚀 Projecting images into 2DPCA space...")
    projected = []
    for i, img in enumerate(images_2d):
        feat = img @ W
        projected.append(feat)
        if i < 3:
            print(f"   🧮 Projected shape for sample {i + 1}: {feat.shape}")
    return projected

def flatten_2dpca_features(projected_images):
    print("\n📦 Flattening projected features for classifier...")
    return np.array([img.flatten() for img in projected_images])

# === RUN 2DPCA ===
num_components = 47
W = compute_2dpca(train_data, num_components)
projected_features = project_2dpca(train_data, W)
flat_features = flatten_2dpca_features(projected_features)

print(f"\n✅ Final Feature Matrix Shape for Classification: {flat_features.shape}")


Test

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

# === CONFIGURATION ===
BASE_PATH = "/content/drive/MyDrive/Datasets/Final UTFVP_Original with Augmented Images_4"
IMAGE_SIZE = (128, 60)
FINGER_NUMS = [1, 2, 3, 4, 5, 6]
TEST_INDICES = [3, 4]  # Original images from img3 and img4

test_data = []
test_labels = []

# === FUNCTION: Denoising ===
def apply_denoising(image, h=10):
    print("🔧 Denoising image...")
    return cv2.fastNlMeansDenoising(image, h=h)

# === FUNCTION TO FUSE FINGERS FOR TESTING ===
def fuse_fingers_test(subject_path, subject_id):
    fused_samples = []
    labels = []

    for img_num in TEST_INDICES:
        finger_images = []
        complete = True
        print(f"\n🧪 Subject {subject_id} — Test Image {img_num} (original)")

        for finger in FINGER_NUMS:
            fname = f"{subject_id}_{finger}_{img_num}.png"
            img_path = os.path.join(subject_path, fname)
            print(f"🖼️ Loading: {img_path}")

            if not os.path.exists(img_path):
                print(f"❌ File not found: {img_path}")
                complete = False
                break

            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, IMAGE_SIZE)
            img_denoised = apply_denoising(img, h=10)
            img_eq = exposure.equalize_hist(img_denoised)
            img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)
            finger_images.append(img_norm)

        if complete and len(finger_images) == 6:
            fused = np.vstack(finger_images)
            label = f"{subject_id}_fused_test_img{img_num}"
            fused_samples.append(fused)
            labels.append(label)
            print(f"✅ Created test fused sample: {label}")
        else:
            print(f"⚠️ Incomplete test sample skipped for Subject {subject_id}, img{img_num}")

    return fused_samples, labels

# === LOAD TEST DATA ===
print("\n🔍 Scanning test subject folders...")
subject_dirs = sorted(os.listdir(BASE_PATH))
for subj in tqdm(subject_dirs, desc="📥 Loading Test Subjects"):
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue
    print(f"\n🔬 Processing test subject: {subj}")
    samples, labels = fuse_fingers_test(subject_path, subj)
    test_data.extend(samples)
    test_labels.extend(labels)

# === PROJECT TEST IMAGES INTO 2DPCA SPACE ===
def project_test_images_2dpca(images_2d, W):
    print("\n📤 Projecting test samples into 2DPCA space...")
    projected = []
    for i, img in enumerate(images_2d):
        feat = img @ W
        projected.append(feat)
        if i < 3:
            print(f"   🧮 Projected shape for test sample {i+1}: {feat.shape}")
    return projected

def flatten_test_features(projected_images):
    return np.array([img.flatten() for img in projected_images])

# === EXECUTE TEST FEATURE EXTRACTION ===
test_data = np.array(test_data)
test_labels = np.array(test_labels)
projected_test = project_test_images_2dpca(test_data, W)
flat_test_data = flatten_test_features(projected_test)

print("\n✅ Final Test Data Shape:", flat_test_data.shape)
print("📌 First Few Test Labels:", test_labels[:5])


Benchmarking

In [ ]:
correct_matches = 0
total_tests = len(flat_test_data)

print("\n📤 Matching test samples using 2DPCA features (Person ID only)...")

# === Step 1: Compare each test sample to all training samples ===
for i in range(total_tests):
    test_vector = flat_test_data[i]
    true_label = test_labels[i]  # e.g., "0001_2_4_3_Augmented"

    # 📏 Compute Manhattan distances to all training vectors
    distances = np.sum(np.abs(flat_features - test_vector), axis=1)

    # 🏆 Nearest neighbor index
    closest_index = np.argmin(distances)
    predicted_label = train_labels[closest_index]

    # 🎯 Extract only subject IDs
    true_id = true_label.split("_")[0]     # e.g., "0001"
    pred_id = predicted_label.split("_")[0]

    # ✅ Person ID match
    if pred_id == true_id:
        match_result = "✅"
        correct_matches += 1
    else:
        match_result = "❌"

    print(f"🔍 Test {i+1:03d}: Predicted = {predicted_label}, Actual = {true_label} {match_result}")

# === Final Accuracy ===
accuracy = (correct_matches / total_tests) * 100
print(f"\n🎉 Person Identification Accuracy: {accuracy:.2f}% ({correct_matches}/{total_tests} correct matches)")
